# MSM vs GMM — Value-at-Risk backtest on the S&P 500

In [1]:
from pathlib import Path
import re
import pandas as pd


def find_project_root(start=None):
    """Walk up from the working dir to the folder containing pyproject.toml."""
    p = (start or Path.cwd()).resolve()
    for d in [p, *p.parents]:
        if (d / "pyproject.toml").exists():
            return d
    raise RuntimeError("project root (pyproject.toml) not found")

PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "artifacts" / "results"

TESTS = [
    "binomial",
    "independence_simple",
    "kupiec",
    "christoffersen_independence",
    "christoffersen_conditional",
]
N_STATES = [2, 3, 4, 5]
MODELS = ["MSM", "GMM"]
DISTRIBUTION = "normal"

## 1. Overview

**Goal.** Compare two models for one-day-ahead **Value-at-Risk (VaR)** on U.S. equities, and test which produces *statistically valid* risk forecasts.

**Models.** Both describe daily log-returns as a **mixture of Gaussians** and take VaR as the α-quantile of that mixture (found by CDF bisection):

- **MarkovSwitchingVaR (MSM)** — an N-state Gaussian **hidden Markov model**: each day the return is drawn from one of N *regimes* (each with its own mean and variance), with a Markov transition matrix governing regime changes. Fit by maximum likelihood (Hamilton filter); the predictive distribution mixes the regimes by their current filtered probabilities. Captures **volatility clustering / regime switching**.
- **GaussianMixtureVaR (GMM)** — a plain N-component Gaussian mixture (scikit-learn), fit by EM on the window. Treats returns as **i.i.d.** draws from a static mixture, with no temporal structure.

The component/emission distribution is always **normal**; the number of regimes/components (`n_states`) ranges over **2–5**.

**Data.** Daily close prices from **Yahoo Finance** for the **S&P 500** constituents with full history over **2010-01-01 → 2026-01-01**. Returns are daily log-returns. The model is refit on a **rolling 252-day window** (~1 trading year); 2010 is **warm-up**, so the backtest covers **15 years (2011–2025)** on **421 stocks**.

**Backtest.** For each day the fitted model predicts VaR at level α; a **breach** (exception) is a day whose realised return falls below the predicted VaR. Over the 15 years we run five standard tests per stock, and report — for each (model, states) — the **share of the 421 stocks that pass** each test at the 5% significance level:

- **binomial** — unconditional coverage: is the breach *frequency* ≈ α? (exact binomial test)
- **independence (simple)** — are breaches serially independent? (χ² on 2×2 transition counts)
- **kupiec** — unconditional coverage via a likelihood-ratio test (breach rate vs α)
- **christoffersen independence** — LR test that a breach is not more likely right after another breach
- **christoffersen conditional** — joint test of correct frequency **and** independence (Kupiec + Christoffersen independence)

**How to run.** One command per configuration, e.g.

```bash
backtester run -m MSM -n 2 -b 2010-01-01 -e 2026-01-01 -w 252 \
    -f features/msm_n2.csv -a 0.05,0.01 -r results/msm_n2.csv
```

The fit phase caches per-window fitted parameters (`features/`, alpha-independent); the predict phase scores each α and writes a per-stock pass/fail table per alpha (`results/`). The tables below aggregate those into pass-rates. See the repository `README.md` for the full option reference.

## 2. Results

### a. α = 0.05

Each cell is **`xx.x% (y)`** — the percentage and count of the 421 stocks that **pass** the test.

In [2]:
def results_table(alpha):
    """Aggregate the per-stock true/false tables into pass-rate cells."""
    rows = []
    for f in sorted(RESULTS_DIR.glob(f"*_a{alpha:g}.csv")):
        m = re.match(r"(msm|gmm)_n(\d+)_a[0-9.]+\.csv", f.name)
        if not m:
            continue
        model, n = m.group(1).upper(), int(m.group(2))
        df = pd.read_csv(f)
        total = len(df)
        rec = {'n_states': n, 'model': model, 'distribution': DISTRIBUTION}
        for t in TESTS:
            passed = int(
                (df[f'{t}_passed'].astype(str).str.lower() == 'true').sum()
            )
            rec[f'{t}_pct'] = f'{100 * passed / total:.1f}% ({passed})'
        rows.append(rec)
    return (
        pd.DataFrame(rows)
        .set_index(['n_states', 'model', 'distribution'])
        .sort_index()
    )

results_table(0.05)

binomial_pct independence_simple_pct   kupiec_pct  \
n_states model distribution                                                     
2        GMM   normal        94.3% (397)               9.5% (40)  94.3% (397)   
         MSM   normal        88.1% (371)             75.5% (318)  88.1% (371)   
3        GMM   normal        96.0% (404)              10.7% (45)  96.0% (404)   
         MSM   normal        67.5% (284)             91.7% (386)  67.5% (284)   
4        GMM   normal        96.2% (405)              11.2% (47)  96.2% (405)   
         MSM   normal           1.0% (4)             57.2% (241)     1.0% (4)   
5        GMM   normal        96.2% (405)              10.9% (46)  96.2% (405)   
         MSM   normal           0.0% (0)               7.4% (31)     0.0% (0)   

                            christoffersen_independence_pct  \
n_states model distribution                                   
2        GMM   normal                             9.5% (40)   
         MSM   normal                           74.1% (312)   
3        GMM   normal                            10.5% (44)   
         MSM   normal                           86.0% (362)   
4        GMM   normal                            10.9% (46)   
         MSM   normal                           46.8% (197)   
5        GMM   normal                            10.9% (46)   
         MSM   normal                             5.5% (23)   

                            christoffersen_conditional_pct  
n_states model distribution                                 
2        GMM   normal                           12.6% (53)  
         MSM   normal                          74.1% (312)  
3        GMM   normal                           13.8% (58)  
         MSM   normal                          67.2% (283)  
4        GMM   normal                           15.2% (64)  
         MSM   normal                             0.7% (3)  
5        GMM   normal                           14.7% (62)  
         MSM   normal                             0.0% (0)

### b. α = 0.01

Same table at the stricter 1% VaR level. Each cell is **`xx.x% (y)`** — the percentage and count of the 421 stocks that **pass** the test.

In [3]:
results_table(0.01)

binomial_pct independence_simple_pct   kupiec_pct  \
n_states model distribution                                                     
2        GMM   normal        42.3% (178)             39.2% (165)  51.1% (215)   
         MSM   normal        45.6% (192)             52.3% (220)  52.3% (220)   
3        GMM   normal        50.4% (212)             41.3% (174)  56.5% (238)   
         MSM   normal        53.2% (224)             74.6% (314)  58.9% (248)   
4        GMM   normal        52.5% (221)             41.6% (175)  59.4% (250)   
         MSM   normal        36.6% (154)             86.5% (364)  41.6% (175)   
5        GMM   normal        53.4% (225)             43.2% (182)  62.9% (265)   
         MSM   normal         16.9% (71)             91.9% (387)   19.5% (82)   

                            christoffersen_independence_pct  \
n_states model distribution                                   
2        GMM   normal                           39.2% (165)   
         MSM   normal                           52.3% (220)   
3        GMM   normal                           41.3% (174)   
         MSM   normal                           74.6% (314)   
4        GMM   normal                           41.6% (175)   
         MSM   normal                           86.2% (363)   
5        GMM   normal                           43.2% (182)   
         MSM   normal                           91.9% (387)   

                            christoffersen_conditional_pct  
n_states model distribution                                 
2        GMM   normal                          28.3% (119)  
         MSM   normal                          36.1% (152)  
3        GMM   normal                          30.9% (130)  
         MSM   normal                          54.2% (228)  
4        GMM   normal                          32.5% (137)  
         MSM   normal                          45.6% (192)  
5        GMM   normal                          34.4% (145)  
         MSM   normal                          25.4% (107)